# Container

Create a file `app.py` with the Python command line application:

In [ ]:
cat ../src/cwl_eoap/crop.py


Create a file called `Dockerfile` with all the necessary commands to assemble an image

In [ ]:
# Build the course wheel in the shared Rocky builder/runtime model.
cat ../Dockerfile


In [ ]:
cat ../Dockerfile

Use `podman` to build the container:

In [ ]:
podman build -t crop -f ../Dockerfile ..

List the container images:

In [5]:
podman images

REPOSITORY                                TAG         IMAGE ID      CREATED                 SIZE
localhost/crop                            latest      c3fd513a0f8e  Less than a second ago  362 MB
docker.io/library/python                  3.10-slim   a28dc131a2de  14 hours ago            127 MB
ghcr.io/eoap/mastering-app-package/stage  1.0.0       cd5590b9b1aa  21 months ago           266 MB
ghcr.io/eoap/quickwin/detect-water-body   <none>      4be723d1a510  22 months ago           530 MB


Run the container to show the `crop` command line tool help:

In [6]:
podman run --rm -it -e PYTHONPATH=/app localhost/crop:latest python -m app --help

Usage: app.py [OPTIONS]

  Crops a STAC Item asset defined with its common band name

Options:
  --input-item TEXT  STAC Item URL or staged STAC catalog  [required]
  --aoi TEXT         Area of interest expressed as a bounding box  [required]
  --epsg TEXT        EPSG code  [required]
  --band TEXT        Common band name  [required]
  --help             Show this message and exit.


Run the application with arguments:

In [ ]:
rm -f crop_green.tif
podman run \
    -i \
    --userns=keep-id \
    --mount=type=bind,source=.,target=/runs \
    --workdir=/runs \
    --read-only=true \
    --user="$(id -u):$(id -g)" \
    --rm \
    --env=HOME=/runs \
    --env=PYTHONPATH=/app \
    localhost/crop:latest \
    python \
    -m \
    app \
    --aoi \
    "-121.399,39.834,-120.74,40.472" \
    --band \
    green \
    --epsg \
    "EPSG:4326" \
    --input-item \
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A &&
python - <<'PY_CHECK'
import rasterio

with rasterio.open("crop_green.tif") as raster:
    assert raster.driver == "GTiff", raster.driver
    assert raster.width > 0 and raster.height > 0 and raster.count > 0
    assert raster.crs is not None, "The cropped raster must have a CRS"
    pixels = raster.read(1, masked=True)
    assert pixels.count() > 0, "The cropped raster contains no valid pixels"
print("Output check passed: readable crop_green.tif with valid pixels and a CRS")
PY_CHECK
